# 18.4 Validating What You Receive — `pydantic`

**Prerequisites:** 18.1 REST and JSON, 16.2 Unions and TypedDict, 8.3 JSON  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- 🔴 **The trust boundary** — data from an API is untrusted input
- What happens without validation: a failure far from its cause
- `BaseModel` — validation that produces a **report**, not one exception
- 🔴 **Coercion vs strict mode** — pydantic changes your data by default
- Constraints with `Annotated` and `Field` — paying off **16.2**
- Nested models, and error paths like `items.1.state`
- 🔴 `TypedDict` vs `dataclass` vs `BaseModel` — when each is right
- Extra fields, aliases, and surviving an API that adds things
- Validating at the boundary and converting to domain objects

---

## 🔴 The trust boundary

**16.1** made the case that annotations are not enforcement, and pointed here for the case where
that matters most:

> **Everything arriving from an API is untrusted input.** Not because the provider is hostile,
> but because you do not control it. It changes, it has bugs, and it has fields your code has
> never seen.

Inside your program, a type checker (**16**) proves things about code *you* wrote. At the edge,
nothing has been proved at all — `response.json()` returns `Any`, and **16.1** showed exactly
how far `Any` spreads.

```
   ┌─── OUTSIDE ────┐   ┌── the boundary ──┐   ┌──── INSIDE ─────┐
   │  JSON, Any,    │──>│    VALIDATE      │──>│ real objects,   │
   │  might be      │   │    once, here    │   │ checked types,  │
   │  anything      │   │                  │   │ trusted         │
   └────────────────┘   └──────────────────┘   └─────────────────┘
```

The fake API below returns data that has *drifted* — every distortion in it is something a real
API has done to somebody.

In [ ]:
# ---- A fake API that returns imperfect data, like real ones do ----
import http.server
import json
import socket
import threading
import urllib.parse

GOOD = {"id": "build-001", "state": "running", "attempts": 2,
        "region": "eu", "owner": {"name": "Aditya", "email": "a@example.com"}}

# 🔴 Every one of these is a real thing an API has done to somebody.
DRIFTED = {"id": 42,                     # was a string yesterday
           "state": "cancelled",         # a state your code has never heard of
           "attempts": "three",          # a number, as a word
           "region": None,               # nullable now, apparently
           "owner": {"name": "Priya"}}   # email missing

MISSING = {"id": "build-003"}            # everything else absent

EXTRA = {**GOOD, "id": "build-004",
         "priority": "high",             # a NEW field the API just added
         "internal_debug_flag": True}


class DriftingAPI(http.server.BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, *args):
        """Silence the default logging."""

    def _send(self, status, payload):
        body = json.dumps(payload).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        path = urllib.parse.urlparse(self.path).path
        routes = {"/jobs/good": GOOD, "/jobs/drifted": DRIFTED,
                  "/jobs/missing": MISSING, "/jobs/extra": EXTRA}
        if path in routes:
            return self._send(200, routes[path])
        if path == "/jobs":
            return self._send(200, {"total": 3,
                                    "items": [GOOD, DRIFTED, MISSING]})
        return self._send(404, {"error": "not found"})


class QuietServer(http.server.ThreadingHTTPServer):
    daemon_threads = True

    def handle_error(self, *args):
        """A client hanging up is normal."""


def start_api():
    probe = socket.socket()
    probe.bind(("127.0.0.1", 0))
    port = probe.getsockname()[1]
    probe.close()
    server = QuietServer(("127.0.0.1", port), DriftingAPI)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{port}"


SERVER, BASE = start_api()
print("fake API on", BASE)
print("endpoints: /jobs/good  /jobs/drifted  /jobs/missing  /jobs/extra  /jobs")

## What happens without validation

The natural code is `response.json()["field"]`. It works until it does not, and then it fails
**far from the cause**.

In [ ]:
import requests

session = requests.Session()


def summarise_naive(payload):
    """Three layers down from the request. Nothing here mentions HTTP."""
    return f"{payload['id']}: {payload['state']}, {payload['attempts'] + 1} attempts next"


for endpoint in ("good", "drifted", "missing"):
    body = session.get(f"{BASE}/jobs/{endpoint}", timeout=5).json()
    try:
        print(f"  /{endpoint:8} -> {summarise_naive(body)}")
    except Exception as exc:
        print(f"  /{endpoint:8} -> 🔴 {type(exc).__name__}: {exc}")

print()
print("🔴 Read the failures. Neither says 'the API returned bad data'.")
print("   One is a TypeError about strings and ints; the other is a KeyError.")
print("   Both happen deep inside your code, with no idea which request")
print("   produced them - the debugging problem 15.9 is about.")

Two different exceptions, neither of which mentions the API. In a real service
this arrives as a `TypeError` in a log at 3 a.m., three call frames from anything that knows a
URL.

**The `drifted` case is worse than the crash**: `attempts` was the string `"three"`. Had the
code done `payload["attempts"]` and stored it, a string would be sitting in your database where
an integer belongs — a **silent** corruption.

## `BaseModel`: validation that reports

```bash
pip install pydantic
```

A pydantic model is a class whose annotations are **enforced at runtime** — the thing **16.1**
said annotations do not do on their own.

In [ ]:
from typing import Literal

from pydantic import BaseModel, ValidationError


class Owner(BaseModel):
    name: str
    email: str


class Job(BaseModel):
    id: str
    state: Literal["queued", "running", "done", "failed"]
    attempts: int
    region: str | None = None
    owner: Owner


good = Job.model_validate(session.get(f"{BASE}/jobs/good", timeout=5).json())
print("valid payload ->", good)
print("   .attempts is a real int:", type(good.attempts).__name__)
print("   .owner is a real Owner :", type(good.owner).__name__, "|", good.owner.name)

print()
print("--- the drifted payload ---")
try:
    Job.model_validate(session.get(f"{BASE}/jobs/drifted", timeout=5).json())
except ValidationError as exc:
    print(f"   {exc.error_count()} problems, ALL of them, in one report:")
    for error in exc.errors():
        location = ".".join(str(part) for part in error["loc"])
        print(f"     {location:16} {error['type']:16} {error['msg'][:46]}")

🔴 **Four errors in one report, each naming its exact location** — including
`owner.email` nested one level down.

That is the difference from hand-written checks: an `if` ladder stops at the first problem, so
you fix one field, redeploy, and discover the next. Pydantic tells you everything that is wrong
in a single pass.

| | Hand-written checks | `BaseModel` |
|---|---|---|
| Reports | the first failure | 🔴 **every** failure |
| Says where | if you wrote a good message | `loc` path, including nesting |
| Converts | you write the `int()` calls | done, with a rule |
| Stays in sync with types | ❌ | 🔴 the annotation **is** the check |

## 🔴 Coercion: pydantic changes your data

By default pydantic runs in **lax mode**: if a value can be sensibly converted to the annotated
type, it converts it. This is genuinely useful at an API boundary — JSON has no integers-vs-
strings discipline — and it is also a surprise the first time.

In [ ]:
from pydantic import ConfigDict


class LaxJob(BaseModel):
    id: str
    attempts: int
    ratio: float


class StrictJob(BaseModel):
    model_config = ConfigDict(strict=True)      # 🔴 no conversions at all
    id: str
    attempts: int
    ratio: float


payload = {"id": "build-1", "attempts": "3", "ratio": "0.5"}

lax = LaxJob.model_validate(payload)
print("lax mode (the default):")
print(f"   attempts {payload['attempts']!r} -> {lax.attempts!r} ({type(lax.attempts).__name__})")
print(f"   ratio    {payload['ratio']!r} -> {lax.ratio!r} ({type(lax.ratio).__name__})")

print()
print("strict mode:")
try:
    StrictJob.model_validate(payload)
except ValidationError as exc:
    for error in exc.errors():
        print(f"   {error['loc'][0]:10} {error['type']:12} {error['msg']}")

print()
print("--- what lax mode will NOT do ---")
for value in ("three", "3.7", True, None):
    try:
        result = LaxJob.model_validate({"id": "b", "attempts": value, "ratio": 1.0})
        print(f"   attempts={value!r:8} -> {result.attempts!r}")
    except ValidationError as exc:
        print(f"   attempts={value!r:8} -> rejected ({exc.errors()[0]['type']})")

Lax mode converted `"3"` to `3` and `"0.5"` to `0.5`, refused `"three"`, and
— note this one — refused `"3.7"` as an `int` rather than silently truncating.

🔴 **`True` was accepted as `1`.** Python's `bool` *is* an `int`, so this is consistent, and it
is also the kind of thing you want to know before it reaches a database.

| Use | When |
|---|---|
| **lax** (default) | 🔴 at an API boundary — JSON types are unreliable and conversion is a feature |
| **strict** | internal data, or where a type change is itself a bug you want reported |

You can also go per-field: `attempts: Strict[int]`, or `Field(strict=True)`.

## Constraints — paying off **16.2**

**16.2** ended with an exercise: build `Annotated[int, Range(0, 10)]` and write a decorator that
enforces it at runtime, noting *"you have just written a tiny pydantic"*. This is the real one.

In [ ]:
from typing import Annotated

from pydantic import Field


class ConstrainedJob(BaseModel):
    id: Annotated[str, Field(min_length=3, max_length=32, pattern=r"^build-\d+$")]
    state: Literal["queued", "running", "done", "failed"]
    attempts: Annotated[int, Field(ge=0, le=10)]
    ratio: Annotated[float, Field(gt=0.0, le=1.0)] = 1.0
    tags: Annotated[list[str], Field(max_length=3)] = []


print("valid:", ConstrainedJob.model_validate(
    {"id": "build-007", "state": "done", "attempts": 3, "ratio": 0.5}))

print()
print("every constraint violated at once:")
try:
    ConstrainedJob.model_validate({
        "id": "nope",                    # wrong pattern, too short
        "state": "done",
        "attempts": 99,                  # above the maximum
        "ratio": 0.0,                    # gt=0 excludes zero
        "tags": ["a", "b", "c", "d"],    # too many
    })
except ValidationError as exc:
    for error in exc.errors():
        location = ".".join(str(p) for p in error["loc"])
        print(f"   {location:10} {error['type']:24} {error['msg'][:44]}")

Each constraint reported separately, by name. `Annotated[int, Field(ge=0,
le=10)]` is exactly the shape **16.2** described — the checker sees `int`, and pydantic reads
the metadata.

| Constraint | For |
|---|---|
| `ge`, `gt`, `le`, `lt` | numbers |
| `min_length`, `max_length` | strings, lists, dicts |
| `pattern` | strings, as a regex (**09**) |
| `multiple_of` | numbers |
| `strict` | disable coercion for this field only |

## Nested models and error paths

Real payloads nest. Pydantic validates all the way down and tells you **where** in the
structure the problem is.

In [ ]:
class JobPage(BaseModel):
    total: int
    items: list[Job]


print("--- validating a whole page, where item 1 is drifted ---")
try:
    JobPage.model_validate(session.get(f"{BASE}/jobs", timeout=5).json())
except ValidationError as exc:
    print(f"   {exc.error_count()} problems:")
    for error in exc.errors():
        location = ".".join(str(part) for part in error["loc"])
        print(f"     {location:22} {error['msg'][:50]}")

print()
print("🔴 `items.1.attempts` names the second item's field. On a 500-item page")
print("   that is the difference between a fix and an afternoon.")

## 🔴 One bad item should not lose the page

The whole page failed because **one** item was malformed. Sometimes that is right — but often
you would rather keep the 499 good records and report the one that failed.

Validate **per item** when partial success is acceptable:

In [ ]:
def parse_page(payload):
    """Return (valid_jobs, problems). One bad item does not lose the rest."""
    valid, problems = [], []
    for index, raw in enumerate(payload["items"]):
        try:
            valid.append(Job.model_validate(raw))
        except ValidationError as exc:
            problems.append((index, raw.get("id", "<no id>"), exc.error_count()))
    return valid, problems


page = session.get(f"{BASE}/jobs", timeout=5).json()
jobs, problems = parse_page(page)

print(f"   kept    : {len(jobs)} valid job(s) -> {[j.id for j in jobs]}")
print(f"   rejected: {len(problems)}")
for index, job_id, count in problems:
    print(f"      item {index} (id={job_id!r}): {count} validation error(s)")

print()
print("🔴 Decide deliberately which you want:")
print("   all-or-nothing  - a payment batch, where partial import is corruption")
print("   best-effort     - a dashboard, where 499 of 500 rows is still useful")
print("   ...and ALWAYS log the rejects (15.10). Silently dropping data is the")
print("   worst of the three, and it is what happens if you forget to look.")

## Extra fields: surviving an API that adds things

APIs add fields. By default pydantic **ignores** unknown ones, which is usually what you
want — your client keeps working when the provider ships a new feature.

| `model_config` | Unknown field | Use when |
|---|---|---|
| `extra="ignore"` (default) | dropped silently | 🔴 **consuming someone else's API** |
| `extra="forbid"` | 🔴 raises | *your own* config files — a typo should be an error |
| `extra="allow"` | kept, accessible | you need to pass unknown data through |

In [ ]:
class IgnoringJob(BaseModel):
    id: str
    state: str


class ForbiddingJob(BaseModel):
    model_config = ConfigDict(extra="forbid")
    id: str
    state: str


class AllowingJob(BaseModel):
    model_config = ConfigDict(extra="allow")
    id: str
    state: str


payload = session.get(f"{BASE}/jobs/extra", timeout=5).json()
print("the API sent:", sorted(payload))

print()
ignored = IgnoringJob.model_validate(payload)
print(f'   extra="ignore" -> {ignored}   (new fields dropped, client survives)')

allowed = AllowingJob.model_validate(payload)
print(f'   extra="allow"  -> priority={allowed.priority!r}, kept for pass-through')

try:
    ForbiddingJob.model_validate(payload)
except ValidationError as exc:
    names = [e["loc"][0] for e in exc.errors()]
    print(f'   extra="forbid" -> rejected: {names}')

print()
print("🔴 forbid is right for YOUR config file (a typo in a key should fail loudly)")
print("   and wrong for someone else's API (their new field should not break you).")

## Aliases: their names, your names

APIs use naming you would not choose — `camelCase`, reserved words, abbreviations. An **alias**
lets the wire format differ from your Python attribute.

In [ ]:
from pydantic import Field


class WireJob(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    job_id: str = Field(alias="jobId")
    is_done: bool = Field(alias="isDone")
    schema_version: int = Field(alias="schema")     # `schema` is awkward in Python


wire = {"jobId": "build-1", "isDone": True, "schema": 3}
parsed = WireJob.model_validate(wire)
print("from the wire:", wire)
print("in Python    :", f"job_id={parsed.job_id!r} is_done={parsed.is_done}"
      f" schema_version={parsed.schema_version}")

print()
print("serialising back out:")
print("   by field name :", parsed.model_dump())
print("   by alias      :", parsed.model_dump(by_alias=True))
print()
print("🔴 model_dump(by_alias=True) is what you send BACK to the API.")
print("   Forgetting it is a common source of 422s on write endpoints.")

## 🔴 `TypedDict` vs `dataclass` vs `BaseModel`

**16.2** compared the first two and gave the rule *"`TypedDict` at the boundary, `dataclass`
inside"*. Pydantic changes that advice, because it validates.

| | `TypedDict` (**16.2**) | `dataclass` (**5.3**) | `BaseModel` |
|---|---|---|---|
| Runtime type | `dict` | a class | a class |
| **Validates at runtime** | ❌ | ❌ | 🔴 **✅** |
| Converts types | ❌ | ❌ | ✅ (lax mode) |
| Serialises to JSON | it *is* a dict | needs `asdict` | `model_dump_json()` |
| Constraints | ❌ | in `__post_init__` | declarative |
| Cost per object | zero | small | 🔴 largest — it does work |
| Best for | describing a shape you already trust | 🔴 **your domain model** | 🔴 **the boundary** |

The revised rule for this folder:

> **`BaseModel` at the boundary. `dataclass` inside.**

Validate once where the data arrives; convert to a plain domain object; let the rest of your
program work with something that cannot be invalid.

In [ ]:
from dataclasses import dataclass


# --- INSIDE: a plain domain object with behaviour, no validation machinery ---
@dataclass(frozen=True, slots=True)
class JobRecord:
    id: str
    state: str
    attempts: int
    owner_name: str

    @property
    def can_retry(self) -> bool:
        return self.state == "failed" and self.attempts < 3


# --- THE BOUNDARY: one function, the only place that trusts nothing ---
class JobPayload(BaseModel):
    id: str
    state: Literal["queued", "running", "done", "failed"]
    attempts: Annotated[int, Field(ge=0)]
    owner: Owner

    def to_record(self) -> JobRecord:
        return JobRecord(id=self.id, state=self.state,
                         attempts=self.attempts, owner_name=self.owner.name)


def fetch_job(session, base, name) -> JobRecord:
    response = session.get(f"{base}/jobs/{name}", timeout=5)
    response.raise_for_status()
    return JobPayload.model_validate(response.json()).to_record()


record = fetch_job(session, BASE, "good")
print("domain object:", record)
print("   can_retry  :", record.can_retry)
print("   frozen     :", end=" ")
try:
    record.attempts = 99
except Exception as exc:
    print(f"{type(exc).__name__} - it cannot be corrupted after validation")

print()
try:
    fetch_job(session, BASE, "drifted")
except ValidationError as exc:
    print(f"bad data fails AT THE BOUNDARY: {exc.error_count()} errors,"
          " before any business logic runs")

Compare that failure with the very first cell of this notebook. Same bad
payload; now it fails **at the boundary**, naming every problem, before a single line of
business logic has run.

And `JobRecord` is `frozen=True` — once validated, it cannot be corrupted.

## Turning a `ValidationError` into something a user can read

If *you* are the API, a `ValidationError` is exactly the `422` body **18.1** described.

In [ ]:
import json as jsonlib


def to_error_response(exc: ValidationError) -> dict:
    """Shape a ValidationError into a 422 body a client can act on."""
    return {
        "error": "validation_failed",
        "detail": [
            {"field": ".".join(str(p) for p in err["loc"]),
             "message": err["msg"],
             "type": err["type"]}
            for err in exc.errors()
        ],
    }


try:
    Job.model_validate(session.get(f"{BASE}/jobs/missing", timeout=5).json())
except ValidationError as exc:
    print("HTTP/1.1 422 Unprocessable Content")
    print(jsonlib.dumps(to_error_response(exc), indent=2)[:520])

print()
print("🔴 Do NOT return the raw str(exc) to a caller: it can include input")
print("   values, which may contain data you do not want echoed back (18.2).")

In [ ]:
# ---- tidy up ----
SERVER.shutdown()
print("fake API stopped")

---

## Common Mistakes & Pitfalls

1. 🔴 **Trusting `response.json()`.** It is `Any` (**16.1**), and `Any` spreads. The failure surfaces far from the API call.
2. 🔴 **Validating deep inside your code instead of at the boundary.** Every layer then has to be defensive, and none of them knows which request caused it.
3. **Being surprised by coercion.** Lax mode converts `"3"` to `3`; that is a feature at a boundary and a bug for internal data. Use `strict=True` when you mean it.
4. 🔴 **Letting one bad item lose an entire page.** Decide deliberately between all-or-nothing and best-effort — and log the rejects either way.
5. **Using `extra="forbid"` on someone else's API.** Their next new field breaks your client for no reason.
6. **Using `extra="ignore"` on your own config file.** A typo'd key is then silently dropped instead of reported.
7. **Forgetting `model_dump(by_alias=True)`** when sending data back, and getting a `422`.
8. **Using `BaseModel` for every internal object.** It validates on every construction; a `dataclass` (**5.3**) is cheaper and enough once the data is trusted.
9. **Returning `str(exc)` to an API caller.** It may echo the input values back (**18.2**).
10. **Treating validation as a substitute for tests.** It checks *shape*, never whether the answer is right (**15.6**, **16.1**).

## Best Practices

- Validate at the boundary, once, in the function that made the request.
- Convert validated payloads into plain domain objects — `frozen=True` dataclasses cannot be corrupted afterwards.
- Keep lax mode at API boundaries; reach for `strict` on internal data.
- Express limits with `Annotated[..., Field(...)]` rather than hand-written `if` checks.
- Use `extra="ignore"` for APIs you consume and `extra="forbid"` for config you own.
- Use aliases so their `camelCase` never leaks into your Python names.
- Log every rejected record with enough context to find it again (**15.10**).
- Report all errors at once — that is the main thing pydantic buys you over `if` ladders.
- Version your models when the API versions its payloads.

## Practice Exercises

Try these before moving on.

1. Point `Job` at `/jobs/extra` and confirm the new fields are ignored. Now switch to `extra="forbid"` and watch it break — which behaviour do you want from *your* API client?
2. 🔴 Add `Field(ge=0, le=10)` to `attempts` and feed it `99`. Compare the error message with what a hand-written `if` would have produced.
3. Write the `if`-ladder equivalent of `Job` by hand. How many lines, and does it report every problem or just the first?
4. Take the `Annotated[int, Range(0, 10)]` exercise from **16.2** and rewrite it with pydantic. How much of your code disappeared?
5. 🔴 Make `parse_page` all-or-nothing, then best-effort, then best-effort-with-logging. Which would you deploy for a payments import, and which for a dashboard?
6. Add a `@field_validator` that rejects a `region` not in a known set, with a message naming the allowed values.
7. Model an API response with `camelCase` keys using aliases, round-trip it with `model_dump(by_alias=True)`, and confirm you get the original JSON back.
8. Benchmark `BaseModel` against a plain `dataclass` for 100,000 constructions (**17.5**). Where would each cost be acceptable?
9. **Interview question:** your API client works for a year, then starts throwing `TypeError` deep in the code. What most likely changed, and what would have caught it at the boundary?

---

## Version notes

| Version | Change |
|---|---|
| **pydantic 2.x** | 🔴 A near-total rewrite with a Rust core — 5–50× faster than v1. `parse_obj` → `model_validate`, `dict()` → `model_dump()`, `Config` class → `model_config = ConfigDict(...)` |
| **pydantic 2.x** | `Annotated[int, Field(...)]` is the recommended form; `x: int = Field(...)` still works |
| **Python 3.11** | `Self` and `assert_never` (**16.2**, **16.4**) compose well with validated models |
| **Python 3.9** | `Annotated` in `typing`, which is what makes the constraint syntax possible |

> **Alternatives.** `attrs` + `cattrs` (mature, more explicit), `msgspec` (very fast, less
> featureful), `marshmallow` (schema-first, pre-dates type hints), or `dataclasses` plus your own
> checks. Pydantic is the default in the ecosystem largely because FastAPI uses it.

## Where next

| Notebook | Covers |
|---|---|
| **18.5** | testing API clients, and concurrency for I/O-bound work |

## Related

- **16.1** — annotations are not enforcement; this is what closes that gap
- **16.2** — `TypedDict`, `Literal` and the `Annotated` exercise this notebook completes
- **8.3 JSON** — the note that pointed here for schema validation
- **5.3 Dataclasses** — the domain object on the inside of the boundary
- **18.1** — the `422` status code a `ValidationError` becomes
- **15.10 Logging** — where rejected records must go